# 문제 정의와 출력 계약 판단

- 기준 TIL: [2026-08-19](../../til/2026/08/2026-08-19.md)
- 관련 강의자료: [머신러닝과 딥러닝의 차이](../../materials/private/kant-deep-learning-basics/01-01_머신러닝과_딥러닝의_차이.md), [딥러닝 문제 유형과 입출력 구조 설계](../../materials/private/kant-deep-learning-basics/01-03_딥러닝_문제_유형과_입출력_구조_설계.md)
- 강의 제공 실습: [01-02 기본](../../materials/private/kant-deep-learning-basics/course-provided-practice/01-02_기본_딥러닝_적용_판단_실습.md), [01-02 심화](../../materials/private/kant-deep-learning-basics/course-provided-practice/01-02_심화_딥러닝_적용_판단_실습.md), [01-04 기본](../../materials/private/kant-deep-learning-basics/course-provided-practice/01-04_기본_문제_유형별_입출력_매핑_실습.md), [01-04 심화](../../materials/private/kant-deep-learning-basics/course-provided-practice/01-04_심화_문제_유형별_입출력_매핑_실습.md)
- 난이도: Core
- 상태: 완료

> 제공 실습의 접근 분류와 입출력 계약 점검 구조를 유지하고, 한 부서 선택이 여러 부서 동시 선택으로 바뀌는 조건까지 실행·검증했습니다.


## 왜 지금 이 실습을 하는가

- TIL에서 확인된 이해: 규칙 기반·머신러닝·딥러닝의 차이를 표현 학습 관점에서 설명했고, 회귀·이진·다중 클래스·다중 레이블의 기본 Shape와 loss를 구분했습니다.
- 이번 실습에서 확인한 부분: 업무 문장을 보고 접근 방식과 `output/target/dtype/loss` 계약을 직접 코드로 만들고, 조건이 바뀌었을 때 계약을 수정한 실행 증거를 남겼습니다.
- 핵심 질문: 문제의 의미에서 출발해 접근 방식과 학습 Tensor 계약을 일관되게 결정할 수 있는가?


## 목표와 완료 기준

- [x] 네 업무 사례의 접근 방식과 task type을 실행 전에 먼저 적었다.
- [x] task별 `output/target/dtype/loss` 계약 생성 함수를 완성했다.
- [x] 올바른 계약과 잘못된 계약을 구분하는 검사를 직접 작성했다.
- [x] `정확히 하나`가 `여러 개 동시 가능`으로 바뀔 때 무엇이 달라지는지 설명했다.
- [x] 이 계약 검사만으로 알 수 없는 한계를 한 가지 적었다.


## 실행 전 예상

아래 표를 코드 실행 전에 채우세요. `Shape`에는 `B`와 `C`를 사용해도 됩니다.

| 업무 | 시작 접근 | task type | output Shape | target Shape/dtype | loss | 판단 근거 |
| --- | --- | --- | --- | --- | --- | --- |
| 가격과 할인율로 최종 가격 계산 | 규칙 기반 | 해당 없음 | - | - | - | 정확한 계산식이 있으므로 학습 모델이 필요하지 않음 |
| 자유 형식 상담 28,000건을 주제별 분류 | 딥러닝 후보 | 다중 클래스 분류 | (B, C) | (B,) / torch.long | CrossEntropyLoss | 자유 형식 표현이 다양하고 라벨 데이터가 충분하지만, 동일한 검증 조건에서 단순 기준선과 비교해야 함 |
| 문서를 네 부서 중 정확히 한 곳으로 전달 | 딥러닝 후보 | 다중 클래스 분류 | (B, 4) | (B,) / torch.long | CrossEntropyLoss | 문서마다 네 부서의 점수를 출력하고 그중 정확히 하나를 선택해야 함 |
| 문서에 환불·불만·개인정보 태그가 동시에 붙을 수 있음 | 딥러닝 후보 | 다중 레이블 분류 | (B, 3) | (B, 3) / torch.float32 | BCEWithLogitsLoss | 태그별로 참과 거짓을 각각 판단하며 여러 태그가 동시에 1일 수 있음 |

특히 두 번째 사례에서는 딥러닝이 `가능한 후보`인 것과 바로 배포할 `최종 선택`인 것을 구분해 적으세요.


In [1]:
# 준비
import torch
from torch import nn

B = 3
C = 4


## 1. task 계약 생성 함수

`regression`, `binary`, `multiclass`, `multilabel`을 입력받아 기본 계약을 반환하는 함수를 작성하세요. 반환값에는 `output_shape`, `target_shape`, `target_dtype`, `loss_name`이 들어가야 합니다.

구현 전에 각 task에서 샘플 하나가 정답을 몇 개 가질 수 있는지 먼저 적으세요.

- `regression`: 샘플마다 연속값 하나
- `binary`: 샘플마다 두 상태 중 하나
- `multiclass`: 샘플마다 여러 class 중 정확히 하나
- `multilabel`: 샘플마다 여러 label이 동시에 참일 수 있음


In [2]:
def contract_for(task, batch_size, class_count=None):
    if task in {"multiclass", "multilabel"} and class_count is None:
        raise ValueError(f"class_count is required for {task}")

    if task == "regression":
        output_shape = (batch_size, 1)
        target_shape = (batch_size, 1)
        target_dtype = torch.float32
        loss_name = "MSELoss"

    elif task == "binary":
        output_shape = (batch_size, 1)
        target_shape = (batch_size, 1)
        target_dtype = torch.float32
        loss_name = "BCEWithLogitsLoss"

    elif task == "multiclass":
        output_shape = (batch_size, class_count)
        target_shape = (batch_size,)
        target_dtype = torch.long
        loss_name = "CrossEntropyLoss"

    elif task == "multilabel":
        output_shape = (batch_size, class_count)
        target_shape = (batch_size, class_count)
        target_dtype = torch.float32
        loss_name = "BCEWithLogitsLoss"

    else:
        raise ValueError(f"unknown task: {task}")

    return {
        "output_shape": output_shape,
        "target_shape": target_shape,
        "target_dtype": target_dtype,
        "loss_name": loss_name,
    }

# class_count가 필요한 task와 필요 없는 task를 구분해 호출합니다.

tasks = ["regression", "binary", "multiclass", "multilabel"]

for task in tasks:
    if task in {"multiclass", "multilabel"}:
        result = contract_for(task, batch_size=3, class_count=3)
    else:
        result = contract_for(task, batch_size=3)

    print(task, result)


regression {'output_shape': (3, 1), 'target_shape': (3, 1), 'target_dtype': torch.float32, 'loss_name': 'MSELoss'}
binary {'output_shape': (3, 1), 'target_shape': (3, 1), 'target_dtype': torch.float32, 'loss_name': 'BCEWithLogitsLoss'}
multiclass {'output_shape': (3, 3), 'target_shape': (3,), 'target_dtype': torch.int64, 'loss_name': 'CrossEntropyLoss'}
multilabel {'output_shape': (3, 3), 'target_shape': (3, 3), 'target_dtype': torch.float32, 'loss_name': 'BCEWithLogitsLoss'}


## 2. loss 직전 계약 검사

`output`과 `target`을 받아 task별 Shape와 dtype을 검사하세요. 함수 안에서 무조건 cast하여 오류를 숨기지 말고, 잘못된 입력은 설명 가능한 메시지와 함께 중단하세요.

최소한 다음을 확인합니다.

- batch 크기가 같은가?
- task에 맞는 rank와 마지막 축인가?
- target dtype이 loss 계약에 맞는가?
- 다중 클래스라면 class index가 범위 안에 있는가?


In [3]:
def validate_contract(task, output, target):
    assert output.shape[0] == target.shape[0], "batch size mismatch"

    if task == "multiclass":
        assert output.ndim == 2 and output.shape[1] > 1, "multiclass output must be (B, C)"
        assert target.ndim == 1, "multiclass target must be (B,)"
        assert output.dtype.is_floating_point, "output must be floating point logits"
        assert target.dtype == torch.long, "target must be torch.long"
        assert int(target.min()) >= 0, "class index must be at least 0"
        assert int(target.max()) < output.shape[1], "class index out of range"

    elif task == "regression":
        assert output.ndim == 2 and output.shape[1] == 1, "regression output must be (B, 1)"
        assert target.ndim == 2 and target.shape[1] == 1, "regression target must be (B, 1)"
        assert output.shape == target.shape, "output/target shape mismatch"
        assert output.dtype.is_floating_point, "output must be floating point"
        assert target.dtype == torch.float32, "target must be torch.float32"

    elif task == "binary":
        assert output.ndim == 2 and output.shape[1] == 1, "binary output must be (B, 1)"
        assert target.ndim == 2 and target.shape[1] == 1, "binary target must be (B, 1)"
        assert output.shape == target.shape, "output/target shape mismatch"
        assert output.dtype.is_floating_point, "output must be floating point logits"
        assert target.dtype == torch.float32, "target must be torch.float32"
        assert torch.all(
            (target == 0) | (target == 1)
        ), "binary target must contain only 0 or 1"

    elif task == "multilabel":
        assert output.ndim == 2 and output.shape[1] > 1, "multilabel output must be (B, C)"
        assert target.ndim == 2, "multilabel target must be (B, C)"
        assert output.shape == target.shape, "output/target shape mismatch"
        assert output.dtype.is_floating_point, "output must be floating point logits"
        assert target.dtype == torch.float32, "target must be torch.float32"
        assert torch.all(
            (target == 0) | (target == 1)
        ), "multilabel target must contain only 0 or 1"

    else:
        raise ValueError(f"unsupported task: {task}")

    return "ok"


normal_cases = [
    ("regression", torch.randn(3, 1), torch.randn(3, 1)),
    ("binary", torch.randn(3, 1), torch.tensor(
        [[0.0], [1.0], [1.0]], dtype=torch.float32
    )),
    ("multiclass", torch.randn(3, 4), torch.tensor([0, 3, 1], dtype=torch.long)),
    ("multilabel", torch.randn(3, 3), torch.tensor(
        [
            [1.0, 1.0, 0.0],
            [0.0, 0.0, 1.0],
            [1.0, 0.0, 1.0],
        ],
        dtype=torch.float32,
    )),
]

for task, output, target in normal_cases:
    print(f"{task}: {validate_contract(task, output, target)}")


invalid_cases = [
    (
        "multiclass target dtype",
        "multiclass",
        torch.randn(3, 4),
        torch.tensor([0.0, 3.0, 1.0]),
    ),
    (
        "multilabel target shape",
        "multilabel",
        torch.randn(3, 3),
        torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]),
    ),
]

for name, task, output, target in invalid_cases:
    try:
        validate_contract(task, output, target)
    except AssertionError as error:
        print(f"{name}: {error}")


regression: ok
binary: ok
multiclass: ok
multilabel: ok
multiclass target dtype: target must be torch.long
multilabel target shape: output/target shape mismatch


## 3. 조건 변경: 한 부서에서 여러 부서로

처음에는 문서마다 네 부서 중 정확히 한 곳만 선택했습니다. 정책이 바뀌어 한 문서가 여러 부서에 동시에 전달될 수 있다고 가정하세요.

실행 전에 다음을 예측하세요.

- task type은 무엇으로 바뀌는가?
  - 다중 레이블
- output과 target Shape은 어떻게 되는가?
  - output은 (B, C) 그대로, target Shape는 기존 (B,)에서 (B, C)로
- target dtype과 loss는 어떻게 바뀌는가?
  - dtype은 torch.long에서 torch.float32로 바뀌고, loss는 CrossEntropyLoss에서 BCEWithLogitsLoss로 바뀜


In [4]:
B, C = 3, 4

logits = torch.randn(B, C)
target = torch.tensor(
    [
        [1.0, 0.0, 1.0, 0.0],
        [0.0, 1.0, 0.0, 1.0],
        [0.0, 0.0, 1.0, 0.0],
    ],
    dtype=torch.float32,
)

loss_fn = nn.BCEWithLogitsLoss()
loss = loss_fn(logits, target)

print("logits:", logits.shape, logits.dtype)
print("target:", target.shape, target.dtype)
print("loss:", loss.shape, loss.dtype)


logits: torch.Size([3, 4]) torch.float32
target: torch.Size([3, 4]) torch.float32
loss: torch.Size([]) torch.float32


## 점진적 힌트

<details>
<summary>힌트 1: 문제 유형부터 막혔을 때</summary>

샘플 하나가 연속값을 내는지, 두 상태 중 하나인지, 여러 class 중 정확히 하나인지, 여러 label이 동시에 참인지 순서대로 확인하세요.
</details>

<details>
<summary>힌트 2: 다중 클래스와 다중 레이블 계약</summary>

둘 다 output의 마지막 축은 class 또는 label 수입니다. 차이는 target이 class index 하나인지, label별 0/1 벡터인지에 있습니다.
</details>

<details>
<summary>힌트 3: 검사 순서</summary>

공통 batch 크기부터 확인한 뒤 rank, 마지막 축, dtype, 값 범위 순으로 좁히면 오류 메시지가 구체적입니다.
</details>


## 결과 해석과 마무리

- 표에서 가장 판단하기 어려웠던 사례와 그 이유: 네 부서 중 하나를 고르는 사례에서 loss 직전 logits `(B, 4)`와 `argmax` 이후 부서 번호 `(B,)`를 구분하기 어려웠다.
- 정상 계약과 실패 계약이 실제로 멈춘 위치: 네 정상 사례는 모두 `ok`였고, 잘못된 다중 클래스 target은 dtype 검사에서, 잘못된 다중 레이블 target은 Shape 검사에서 멈췄다.
- `정확히 한 부서`에서 `여러 부서 동시 가능`으로 바뀔 때 달라진 항목: output은 `(B, C)`로 같지만 target은 `(B,)` long에서 `(B, C)` float32로, loss는 `CrossEntropyLoss`에서 `BCEWithLogitsLoss`로 바뀐다.
- Shape와 dtype 검사가 맞아도 보장하지 못하는 것 한 가지: class index와 실제 부서 이름의 매핑이 올바른지는 보장하지 못한다.
- 딥러닝 후보를 선택했더라도 배포 전에 단순 기준선과 같은 검증 조건에서 비교해야 하는 이유: 딥러닝이 실제 정확도 개선을 만드는지와 지연시간·운영 비용을 함께 확인해야 최종 선택을 정할 수 있기 때문이다.
